# Bahdanau NumPy summarizer — inference only

Loads a saved checkpoint produced by `bahdanau_numpy_rnn.ipynb` and generates a
summary for any text you paste in. No training loop, no BBC News Summary dataset
loading required - just the model class definitions (copied verbatim) plus a
checkpoint file.

**Which checkpoint:** defaults to the **overfit sanity-check model**
(`bahdanau_numpy_overfit.npz` / `_meta.json`, from section 8.5 of the main notebook) -
the current, non-backup checkpoint on disk. Its vocabulary is only the ~2000 words
from the 100 articles it was trained on, and it was deliberately trained to
*memorize* those examples, not generalize - so it reproduces its own training
articles almost perfectly, but will be mostly `<UNK>`-heavy garbage on genuinely new
text (paste one of the "Article:" lines printed by section 8.5 in the main notebook
to see it work well).

Once the full-vocab model (16081 words, 1557 examples) is retrained and saved fresh
under its plain (non-backup) filename, change `WEIGHTS_PATH`/`META_PATH` in the
"Load checkpoint" cell below to point at it instead - that's the one meant for
real-world text.

In [23]:
import os
import json
import numpy as np

DTYPE = np.float32
MAX_ARTICLE_LEN = 40  # must match max_article_len used to preprocess training data


## Model definition (copied verbatim from `bahdanau_numpy_rnn.ipynb`)

In [24]:
def xavier(shape, dtype=DTYPE):
    fan_out = shape[0]
    fan_in = shape[1] if len(shape) > 1 else shape[0]
    limit = np.sqrt(6.0 / (fan_in + fan_out))
    return np.random.uniform(-limit, limit, size=shape).astype(dtype)


def softmax(x):
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)


In [25]:
class Embedding:
    """NumPy analogue of nn.Embedding: a (vocab, emb) lookup table."""

    def __init__(self, vocab_size, emb_dim, dtype=DTYPE):
        self.W = xavier((vocab_size, emb_dim), dtype)
        self.dW = np.zeros_like(self.W)

    def forward(self, idx):
        return self.W[idx]

    def backward(self, idx, dvec):
        self.dW[idx] += dvec

    def zero_grad(self):
        self.dW.fill(0)

    def parameters(self):
        return [(self.W, self.dW)]

    def named_parameters(self, prefix=""):
        return [(prefix + "W", self.W)]


In [26]:
class Linear:
    """NumPy analogue of nn.Linear: out = W @ x + b."""

    def __init__(self, in_dim, out_dim, dtype=DTYPE):
        self.W = xavier((out_dim, in_dim), dtype)
        self.b = np.zeros(out_dim, dtype=dtype)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, x):
        out = self.W @ x + self.b
        return out, x  # cache = input x

    def backward(self, dout, cache):
        x = cache
        self.dW += np.outer(dout, x)
        self.db += dout
        return self.W.T @ dout

    def zero_grad(self):
        self.dW.fill(0)
        self.db.fill(0)

    def parameters(self):
        return [(self.W, self.dW), (self.b, self.db)]

    def named_parameters(self, prefix=""):
        return [(prefix + "W", self.W), (prefix + "b", self.b)]


In [27]:
class RNNCell:
    """Vanilla tanh RNN cell (used in place of nn.GRU):
    h_t = tanh(Wxh @ x_t + Whh @ h_{t-1} + bh)
    """

    def __init__(self, input_size, hidden_size, dtype=DTYPE):
        self.Wxh = xavier((hidden_size, input_size), dtype)
        self.Whh = xavier((hidden_size, hidden_size), dtype)
        self.bh = np.zeros(hidden_size, dtype=dtype)
        self.dWxh = np.zeros_like(self.Wxh)
        self.dWhh = np.zeros_like(self.Whh)
        self.dbh = np.zeros_like(self.bh)

    def step(self, x_t, h_prev):
        z = self.Wxh @ x_t + self.Whh @ h_prev + self.bh
        h_t = np.tanh(z)
        return h_t, (x_t, h_prev, h_t)

    def step_backward(self, dh_t, cache):
        x_t, h_prev, h_t = cache
        dz = dh_t * (1 - h_t ** 2)                 # tanh backward
        self.dWxh += np.outer(dz, x_t)
        self.dWhh += np.outer(dz, h_prev)
        self.dbh += dz
        dx_t = self.Wxh.T @ dz
        dh_prev = self.Whh.T @ dz
        return dx_t, dh_prev

    def zero_grad(self):
        self.dWxh.fill(0)
        self.dWhh.fill(0)
        self.dbh.fill(0)

    def parameters(self):
        return [(self.Wxh, self.dWxh), (self.Whh, self.dWhh), (self.bh, self.dbh)]

    def named_parameters(self, prefix=""):
        return [(prefix + "Wxh", self.Wxh), (prefix + "Whh", self.Whh), (prefix + "bh", self.bh)]


In [28]:
class BahdanauAttention:
    def __init__(self, hidden_size, dtype=DTYPE):
        self.W_s = xavier((hidden_size, hidden_size), dtype)
        self.b_s = np.zeros(hidden_size, dtype=dtype)
        self.W_h = xavier((hidden_size, hidden_size), dtype)
        self.b_h = np.zeros(hidden_size, dtype=dtype)
        self.v = xavier((1, hidden_size), dtype)
        self.bv = np.zeros(1, dtype=dtype)

        self.dW_s = np.zeros_like(self.W_s)
        self.db_s = np.zeros_like(self.b_s)
        self.dW_h = np.zeros_like(self.W_h)
        self.db_h = np.zeros_like(self.b_h)
        self.dv = np.zeros_like(self.v)
        self.dbv = np.zeros_like(self.bv)

    def forward(self, decoder_hidden, encoder_outputs):
        # U[i] = tanh(W_s @ decoder_hidden + b_s + W_h @ encoder_outputs[i] + b_h)
        query_proj = self.W_s @ decoder_hidden + self.b_s          # (H,)
        keys_proj = encoder_outputs @ self.W_h.T + self.b_h        # (T_x, H)
        U = np.tanh(keys_proj + query_proj)                        # (T_x, H)
        scores = U @ self.v[0] + self.bv[0]                        # (T_x,)
        alpha = softmax(scores)                                    # (T_x,)
        context = alpha @ encoder_outputs                          # (H,)
        cache = (U, alpha, encoder_outputs, decoder_hidden)
        return context, alpha, cache

    def backward(self, dcontext, cache):
        U, alpha, H_enc, decoder_hidden = cache

        dH_enc = np.outer(alpha, dcontext)                         # context path
        dalpha = H_enc @ dcontext
        dscores = alpha * (dalpha - np.sum(alpha * dalpha))        # softmax backward

        self.dv[0] += dscores @ U
        self.dbv[0] += np.sum(dscores)
        dU = np.outer(dscores, self.v[0])
        dz = dU * (1 - U ** 2)                                     # tanh backward

        dz_sum = dz.sum(axis=0)
        self.dW_s += np.outer(dz_sum, decoder_hidden)
        self.db_s += dz_sum
        self.dW_h += dz.T @ H_enc
        self.db_h += dz_sum

        ddecoder_hidden = self.W_s.T @ dz_sum
        dH_enc += dz @ self.W_h                                    # attention-key path
        return ddecoder_hidden, dH_enc

    def zero_grad(self):
        for g in (self.dW_s, self.db_s, self.dW_h, self.db_h, self.dv, self.dbv):
            g.fill(0)

    def parameters(self):
        return [(self.W_s, self.dW_s), (self.b_s, self.db_s),
                (self.W_h, self.dW_h), (self.b_h, self.db_h),
                (self.v, self.dv), (self.bv, self.dbv)]

    def named_parameters(self, prefix=""):
        return [(prefix + "W_s", self.W_s), (prefix + "b_s", self.b_s),
                (prefix + "W_h", self.W_h), (prefix + "b_h", self.b_h),
                (prefix + "v", self.v), (prefix + "bv", self.bv)]


In [29]:
class Encoder:
    def __init__(self, embedding, emb_dim, hidden_size, dtype=DTYPE):
        self.embedding = embedding
        self.rnn_cell = RNNCell(emb_dim, hidden_size, dtype)
        self.hidden_size = hidden_size
        self.dtype = dtype

    def forward(self, x_ids):
        T = len(x_ids)
        H = np.zeros((T, self.hidden_size), dtype=self.dtype)
        caches = []
        h_prev = np.zeros(self.hidden_size, dtype=self.dtype)
        for t in range(T):
            x_t = self.embedding.forward(x_ids[t])
            h_t, rnn_cache = self.rnn_cell.step(x_t, h_prev)
            H[t] = h_t
            caches.append((x_ids[t], rnn_cache))
            h_prev = h_t
        return H, caches

    def backward(self, dH, dh_last, caches):
        """dH: gradient into every H[t] from attention. dh_last: gradient into
        H[-1] from being used as the decoder's initial hidden state (the bridge)."""
        dh_next = dh_last
        for t in reversed(range(len(caches))):
            x_id, rnn_cache = caches[t]
            dh_total = dH[t] + dh_next
            dx_t, dh_prev = self.rnn_cell.step_backward(dh_total, rnn_cache)
            self.embedding.backward(x_id, dx_t)
            dh_next = dh_prev

    def zero_grad(self):
        self.rnn_cell.zero_grad()

    def parameters(self):
        return self.rnn_cell.parameters()

    def named_parameters(self, prefix=""):
        return self.rnn_cell.named_parameters(prefix + "rnn_cell.")


In [30]:
class Decoder:
    def __init__(self, embedding, emb_dim, hidden_size, vocab_size, dtype=DTYPE):
        self.embedding = embedding
        self.attention = BahdanauAttention(hidden_size, dtype)
        self.rnn_cell = RNNCell(emb_dim + hidden_size, hidden_size, dtype)
        self.fc_out = Linear(hidden_size * 2 + emb_dim, vocab_size, dtype)
        self.hidden_size = hidden_size
        self.emb_dim = emb_dim

    def step(self, input_token_id, hidden, encoder_outputs):
        embedded = self.embedding.forward(input_token_id)
        context, alpha, attn_cache = self.attention.forward(hidden, encoder_outputs)
        rnn_input = np.concatenate([embedded, context])
        hidden_new, rnn_cache = self.rnn_cell.step(rnn_input, hidden)
        pred_input = np.concatenate([hidden_new, context, embedded])
        logits, fc_cache = self.fc_out.forward(pred_input)
        cache = (input_token_id, attn_cache, rnn_cache, fc_cache)
        return logits, hidden_new, alpha, cache

    def step_backward(self, dlogits, dhidden_next, cache):
        input_token_id, attn_cache, rnn_cache, fc_cache = cache
        H, E = self.hidden_size, self.emb_dim

        dpred_input = self.fc_out.backward(dlogits, fc_cache)
        dhidden_new = dpred_input[:H] + dhidden_next
        dcontext = dpred_input[H:2 * H]
        dembedded = dpred_input[2 * H:2 * H + E].copy()

        drnn_input, dhidden_prev = self.rnn_cell.step_backward(dhidden_new, rnn_cache)
        dembedded += drnn_input[:E]
        dcontext = dcontext + drnn_input[E:]

        ddecoder_hidden, dH_enc = self.attention.backward(dcontext, attn_cache)
        dhidden_prev = dhidden_prev + ddecoder_hidden

        self.embedding.backward(input_token_id, dembedded)
        return dhidden_prev, dH_enc

    def zero_grad(self):
        self.attention.zero_grad()
        self.rnn_cell.zero_grad()
        self.fc_out.zero_grad()

    def parameters(self):
        return self.attention.parameters() + self.rnn_cell.parameters() + self.fc_out.parameters()

    def named_parameters(self, prefix=""):
        params = []
        params += self.attention.named_parameters(prefix + "attention.")
        params += self.rnn_cell.named_parameters(prefix + "rnn_cell.")
        params += self.fc_out.named_parameters(prefix + "fc_out.")
        return params


In [31]:
class Seq2Seq:
    def __init__(self, vocab_size, emb_dim, hidden_size, dtype=DTYPE):
        self.embedding = Embedding(vocab_size, emb_dim, dtype)  # shared, like torch's shared_embedding
        self.encoder = Encoder(self.embedding, emb_dim, hidden_size, dtype)
        self.decoder = Decoder(self.embedding, emb_dim, hidden_size, vocab_size, dtype)

    def greedy_decode(self, src_ids, sos_id, eos_id, max_len):
        H, _ = self.encoder.forward(src_ids)
        hidden = H[-1]
        token = sos_id
        tokens = []
        for _ in range(max_len):
            logits, hidden, alpha, _ = self.decoder.step(token, hidden, H)
            token = int(np.argmax(logits))
            if token == eos_id:
                break
            tokens.append(token)
        return tokens

    def zero_grad(self):
        self.embedding.zero_grad()
        self.encoder.zero_grad()
        self.decoder.zero_grad()

    def parameters(self):
        return self.embedding.parameters() + self.encoder.parameters() + self.decoder.parameters()

    def named_parameters(self):
        params = []
        params += self.embedding.named_parameters("embedding.")
        params += self.encoder.named_parameters("encoder.")
        params += self.decoder.named_parameters("decoder.")
        return params

    def state_dict(self):
        return {name: arr.copy() for name, arr in self.named_parameters()}

    def load_state_dict(self, state_dict):
        own_params = dict(self.named_parameters())
        assert set(own_params) == set(state_dict), "checkpoint parameter names do not match model"
        for name, arr in state_dict.items():
            own_params[name][...] = arr


## Load checkpoint

`greedy_decode` is the only inference method needed, so `forward`/`backward`
(training-only) are omitted here from `Seq2Seq` - the layer classes above keep theirs
since `load_state_dict` doesn't care and it's simplest to copy them verbatim.

In [32]:
def load_checkpoint(weights_path, meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    loaded_stoi = meta["stoi"]
    loaded_itos = {int(i): w for w, i in loaded_stoi.items()}
    loaded_model = Seq2Seq(meta["VOCAB_SIZE"], meta["EMBEDDING_SIZE"], meta["HIDDEN_SIZE"], dtype=DTYPE)
    data = np.load(weights_path)
    loaded_model.load_state_dict({k: data[k] for k in data.files})
    return loaded_model, loaded_stoi, loaded_itos, meta["TARGET_LEN"]


CHECKPOINT_DIR = "../DL/checkpoints"
WEIGHTS_PATH = os.path.join(CHECKPOINT_DIR, "bahdanau_numpy_overfit.npz")
META_PATH = os.path.join(CHECKPOINT_DIR, "bahdanau_numpy_overfit_meta.json")

model, stoi, itos, TARGET_LEN = load_checkpoint(WEIGHTS_PATH, META_PATH)
print(f"Loaded checkpoint from {WEIGHTS_PATH}")
print(f"Vocab size: {len(stoi)}, TARGET_LEN: {TARGET_LEN}")


Loaded checkpoint from ../DL/checkpoints/bahdanau_numpy_overfit.npz
Vocab size: 2050, TARGET_LEN: 12


## Proof it actually learned: reproduce its own training data

These are real `(article, summary)` pairs from the overfit model's own 100-example
training set (hardcoded here so this notebook doesn't need to reload the BBC
dataset - see section 8.5 of `bahdanau_numpy_rnn.ipynb` for where they came from).
The model was deliberately trained to *memorize* these, so it should reproduce each
summary essentially exactly. This is the actual evidence that the architecture and
hand-derived backprop learned to summarize - as opposed to the `<UNK>`-heavy output
you'll see below on genuinely unseen text, which is a vocabulary/generalization
limit of this small sanity-check checkpoint, not a broken pipeline.

In [33]:
def encode_custom_article(text, stoi_map, max_len):
    words = text.lower().split()[:max_len]
    ids = np.array([stoi_map.get(w, stoi_map["<UNK>"]) for w in words], dtype=np.int64)
    return ids, words


# (article, summary) pairs the overfit model was actually trained on
known_training_examples = [
    ("surfers outside the us have been unable to visit the official re-election site "
     "of president george w bush. the blocking of browsers sited outside the us began "
     "in the early hours of monday morning. since then people outside the us",
     "since then people outside the us trying to browse the"),
    ("eurosceptic party ukip have suspended a candidate for allegedly suggesting the "
     "criminally insane should be killed. john houston, 54, was due to stand in the "
     "east kilbride seat in lanarkshire at the next election. but he was suspended after his",
     "peter nielson, who is ukip scotland chairman, said he had"),
    ("nearly 20% more uk top 250 firms produced non-financial reports on social and "
     "environment issues than last year. but of the 145 companies reporting, 76% didn't "
     "examine their supply chains, says the annual directions survey. green groups say putting pressure",
     "less than a quarter of companies (24%) get their corporate"),
]

for article, actual_summary in known_training_examples:
    src_ids, kept_words = encode_custom_article(article, stoi, MAX_ARTICLE_LEN)
    n_unk = int(np.sum(src_ids == stoi["<UNK>"]))
    tokens = model.greedy_decode(src_ids, stoi["<SOS>"], stoi["<EOS>"], max_len=TARGET_LEN)
    predicted = " ".join(itos[t] for t in tokens if t not in (stoi["<SOS>"], stoi["<EOS>"], stoi["<PAD>"]))
    match = "exact match" if predicted == actual_summary else "differs"
    print(f"Article:   {article}")
    print(f"Actual:    {actual_summary}")
    print(f"Predicted: {predicted}   [{match}]")
    print("=" * 60)


Article:   surfers outside the us have been unable to visit the official re-election site of president george w bush. the blocking of browsers sited outside the us began in the early hours of monday morning. since then people outside the us
Actual:    since then people outside the us trying to browse the
Predicted: since then people outside the us trying to browse the   [exact match]
Article:   eurosceptic party ukip have suspended a candidate for allegedly suggesting the criminally insane should be killed. john houston, 54, was due to stand in the east kilbride seat in lanarkshire at the next election. but he was suspended after his
Actual:    peter nielson, who is ukip scotland chairman, said he had
Predicted: peter nielson, who is ukip scotland chairman, said he had   [exact match]
Article:   nearly 20% more uk top 250 firms produced non-financial reports on social and environment issues than last year. but of the 145 companies reporting, 76% didn't examine their supply chains, says

## Summarize your own text

Edit `custom_article` and re-run. Text is lowercased and truncated to
`MAX_ARTICLE_LEN` words, matching training preprocessing. Words outside the loaded
checkpoint's vocabulary become `<UNK>` - the cell reports how many.

In [35]:
custom_article = """
eurosceptic party ukip have suspended a candidate for allegedly suggesting the criminally insane should be killed.
"""

src_ids, kept_words = encode_custom_article(custom_article, stoi, MAX_ARTICLE_LEN)
n_unk = int(np.sum(src_ids == stoi["<UNK>"]))

tokens = model.greedy_decode(src_ids, stoi["<SOS>"], stoi["<EOS>"], max_len=TARGET_LEN)
summary = " ".join(itos[t] for t in tokens if t not in (stoi["<SOS>"], stoi["<EOS>"], stoi["<PAD>"]))

print(f"Article (truncated to {len(kept_words)}/{MAX_ARTICLE_LEN} words, {n_unk} unknown to vocab):")
print(" ".join(kept_words))
print()
print("Generated summary:")
print(summary)


Article (truncated to 16/40 words, 0 unknown to vocab):
eurosceptic party ukip have suspended a candidate for allegedly suggesting the criminally insane should be killed.

Generated summary:
peter nielson, who is ukip scotland chairman, said he had
